<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/hmu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = "KGAT_c03d989b55c966d18c971a92b023645b"

!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:06<00:00, 31.5MB/s]



In [ ]:
!unzip -q busi-dataset.zip -d busi_dataset

In [ ]:
!pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00


In [ ]:
import os
import copy
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from PIL import Image
import albumentations as A
from tqdm import tqdm

# Configuration
BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 8
SEED = 42
EPOCHS = 10

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class DSConv2d(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, padding=1, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.pointwise(self.depthwise(x))))

class LightDCSAM(nn.Module):

    def __init__(self, channels):
        super(LightDCSAM, self).__init__()
        self.smooth = nn.AvgPool2d(3, stride=1, padding=1)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // 4, 4), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // 4, 4), channels, 1, bias=False),
            nn.Sigmoid()
        )

        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 3, padding=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        details = x - self.smooth(x)
        x = x + details
        x = x * self.ca(x)
        sa_in = torch.cat([torch.mean(x, dim=1, keepdim=True), torch.max(x, dim=1, keepdim=True)[0]], dim=1)
        return x * self.sa(sa_in)

class LightHMAM(nn.Module):
    """ Optimized Hybrid Multi-scale Attention """
    def __init__(self, channels):
        super(LightHMAM, self).__init__()
        inter = channels // 4

        self.reduce = nn.Conv2d(channels, inter * 3, 1, bias=False)

        self.d1 = nn.Conv2d(inter, inter, 3, padding=1, dilation=1, groups=inter, bias=False)
        self.d3 = nn.Conv2d(inter, inter, 3, padding=3, dilation=3, groups=inter, bias=False)
        self.d5 = nn.Conv2d(inter, inter, 3, padding=5, dilation=5, groups=inter, bias=False)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pool_conv = nn.Conv2d(channels, inter, 1, bias=False)

        self.fuse = nn.Conv2d(inter * 4, channels, 1, bias=False)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 4, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b1, b2, b3 = torch.split(self.reduce(x), x.shape[1] // 4, dim=1)
        b1, b2, b3 = self.d1(b1), self.d3(b2), self.d5(b3)
        b4 = F.interpolate(self.pool_conv(self.pool(x)), size=x.shape[2:], mode='bilinear')

        fused = self.fuse(torch.cat([b1, b2, b3, b4], dim=1))
        return fused * self.se(fused)

In [ ]:
class DoubleDSConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            DSConv2d(in_channels, out_channels),
            DSConv2d(out_channels, out_channels)
        )
    def forward(self, x):
        return self.net(x)

class LightHMUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(LightHMUNet, self).__init__()

        self.enc1 = DoubleDSConv(in_channels, 16)
        self.enc2 = DoubleDSConv(16, 32)
        self.enc3 = DoubleDSConv(32, 64)
        self.enc4 = DoubleDSConv(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.dcsam1 = LightDCSAM(16)
        self.dcsam2 = LightDCSAM(32)
        self.dcsam3 = LightDCSAM(64)
        self.dcsam4 = LightDCSAM(128)

        self.bottleneck = DoubleDSConv(128, 256)
        self.hmam = LightHMAM(256)

        self.up4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec4 = DoubleDSConv(256, 128)

        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = DoubleDSConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = DoubleDSConv(64, 32)

        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = DoubleDSConv(32, 16)

        self.final_conv = nn.Conv2d(16, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.hmam(self.bottleneck(self.pool(e4)))

        d4 = self.dec4(torch.cat([self.dcsam4(e4), self.up4(b)], dim=1))
        d3 = self.dec3(torch.cat([self.dcsam3(e3), self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([self.dcsam2(e2), self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([self.dcsam1(e1), self.up1(d2)], dim=1))

        return self.final_conv(d1)

In [ ]:
class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if len(mask_files) == 0: continue
                mask_paths = [os.path.join(cls_dir, f) for f in mask_files]
                self.samples.append((img_path, mask_paths))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            mask = (mask > 0).astype(np.uint8)
            combined_mask = np.logical_or(combined_mask, mask)
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image = augmented["image"]
            combined_mask = augmented["mask"]

        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(combined_mask).unsqueeze(0).float()
        return image, mask

val_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)])
train_transform = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=0.5), A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)])

full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)

train_size, val_size = int(0.8 * len(full_dataset)), int(0.1 * len(full_dataset))
train_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform), indices[val_size:train_size + val_size])
val_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[:val_size])
test_dataset = Subset(BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform), indices[train_size + val_size:])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        preds = torch.sigmoid(logits).view(-1)
        targets_f = targets.view(-1)
        inter = (preds * targets_f).sum()
        dice_loss = 1 - (2 * inter + self.smooth) / (preds.sum() + targets_f.sum() + self.smooth)
        return 0.5 * bce_loss + 0.5 * dice_loss

def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f, y_pred_f = y_true.view(-1), y_pred.view(-1)
    inter = (y_true_f * y_pred_f).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

model = LightHMUNet(in_channels=3, out_channels=1).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

best_val_loss = float("inf")
best_model_weights = None

for epoch in range(EPOCHS):
    model.train()
    for images, masks in tqdm(train_loader):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    model.eval()
    val_loss = val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, masks)
            val_loss += loss.item()
            val_dice += dice_coef(masks, torch.sigmoid(logits)).item()

    avg_val_loss, avg_val_dice = val_loss / len(val_loader), val_dice / len(val_loader)
    print(f"{avg_val_loss:.4f} {avg_val_dice:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_light_hmunet.pth")

model.load_state_dict(best_model_weights)
model.eval()
test_dice = 0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            logits = model(images)
        test_dice += dice_coef(masks, torch.sigmoid(logits)).item()

print(f"{test_dice / len(test_loader):.4f}")

/tmp/ipykernel_5293/2970865052.py:74: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/65 [00:00<?, ?it/s]/tmp/ipykernel_5293/2970865052.py:85: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 65/65 [00:49<00:00,  1.33it/s]
/tmp/ipykernel_5293/2970865052.py:98: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


0.7722 0.0000


100%|██████████| 65/65 [00:07<00:00,  8.74it/s]


0.7030 0.0000


100%|██████████| 65/65 [00:06<00:00, 10.77it/s]


...

0.5587 0.0000


100%|██████████| 65/65 [00:06<00:00, 10.72it/s]


0.5458 0.0000


/tmp/ipykernel_5293/2970865052.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


0.0251
